In [1]:
using LogDensityProblems: LogDensityProblems;
using Distributions
using DelimitedFiles
using Random: Random

using MCMCChains

In [2]:
n_schools = 8
y = [28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0] # estimated treatment effects
σ = [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0]
# Let's define some type that represents the model.
struct RegressionProblem{Ty <: AbstractVector}
	y::Ty
	σ::Ty
end
LogDensityProblems.dimension(model::RegressionProblem) = 10

function LogDensityProblems.logdensity(model::RegressionProblem, parameters::AbstractVector{<:Real})
	μ,τ,θ = parameters[1],parameters[2],parameters[3:end]
	lp = logpdf(Normal(0, 5), μ)
	lp += logpdf(truncated(Cauchy(0, 5),lower=0), τ)

	for i in 1:8
		lp += logpdf(Normal(0, 1), θ[i])
		lp += logpdf(Normal(μ + τ * θ[i], model.σ[i]), model.y[i])
	end
	return lp
end

LogDensityProblems.capabilities(model::RegressionProblem) = LogDensityProblems.LogDensityOrder{0}()
MYmodel = RegressionProblem(y, σ)

RegressionProblem{Vector{Float64}}([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0], [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])

In [3]:
function tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	N_e = max(1, N_e)

	if t <= M_adapt
		return 2μ * N_e / (N_e + N_c)
	else
		return μ
	end
end

function get_complementary(i, N)
	indices = collect(1:N)
	deleteat!(indices, i)
	return indices
end

function get_direction_vector(S, l, m, μ)
	return μ * (S[l, :] - S[m, :])
end

function DifferentialMove(rng, k, μ, S, N)
	# work on walker k
	indices = get_complementary(k, N)
	# draw two random indices from the complementary set, without replacement
	l, m = sample(rng, indices, 2, replace = false)
	return get_direction_vector(S, l, m, μ)
end

DifferentialMove

In [10]:
using AbstractMCMC

struct EnsembleSliceSampler{T<:Float64,A<:Int64} <: AbstractMCMC.AbstractSampler
    "initial length scale"
    μ_init::T
    "number of adapation steps"
    M_adapt::A
    "number of walkers"
    N_walkers::A
    "max number of attempts"
    max_steps::A
end

struct ESState{A<:AbstractMatrix{<:Real},T<:Float64,B<:Int64}
    "current position"
    x::A
    "length scale"
    μ::T
    "iteration"
    t::B
end

struct ESSample{A<:AbstractMatrix{<:Real}}
    "current position"
    x::A # a matrix of dimension n_walkers * n_params
end

In [11]:
function AbstractMCMC.step(
	rng::Random.AbstractRNG,
	model_wrapper::AbstractMCMC.LogDensityModel,
	sampler::EnsembleSliceSampler,
	state::ESState)

	model = model_wrapper.logdensity
	# extract the sampler parameters
	μ = sampler.μ_init
	M_adapt = sampler.M_adapt
	N_walkers = sampler.N_walkers
	max_steps = sampler.max_steps
	f(x) = LogDensityProblems.logdensity(model, x)

	# extract current state
	x, μ, t = state.x, state.μ, state.t
	N_dim = size(x, 2)

	x_new = Matrix{Float64}(undef, N_walkers, N_dim)

	R, L, N_e, N_c = 0, 0, 0, 0
	X′ = 0

	# loop over the walkers
	for k in 1:N_walkers

		Xₖ = x[k, :] # get the current position of walker k
		ηₖ = DifferentialMove(rng, k, μ, x, N_walkers) # get the differential move

		δ = rand(rng, Exponential(1))
		Y = f(Xₖ) - δ

		L = -rand(rng)
		R = L + 1
		l = 0
		while Y < f(L .* ηₖ + Xₖ)
			L = L - 1
			N_e = N_e + 1
			l += 1
			if l == max_steps
				println("L: ", L, " Y: ", Y, " f(L): ", f(L .* ηₖ + Xₖ))
				error("Max steps reached", " iteration: ", t, " walker: ", k)
			end
		end
		l = 0
		while Y < f(R .* ηₖ + Xₖ)
			R = R + 1
			N_e = N_e + 1
			l += 1
			if l == max_steps
				println("L: ", R, " Y: ", Y, " f(R): ", f(R .* ηₖ + Xₖ))
				error("Max steps reached")
			end
		end

		l = 0
		while true
			l += 1
			X′ = rand(rng, Uniform(L, R))
			Y′ = f(X′ .* ηₖ + Xₖ)
			if Y < Y′
				break
			end
			if X′ < 0
				L = X′
				N_c = N_c + 1
			else
				R = X′
				N_c = N_c + 1
			end
			if l == max_steps
				println("L: ", R, " Y: ", Y, " f(R): ", f(R .* ηₖ + Xₖ))

				error("Max steps reached")
			end
		end

		Xₖ = X′ .* ηₖ + Xₖ
		x_new[k, :] = Xₖ
	end
	# println("R: ", R, " L: ", L, " N_e: ", N_e, " N_c: ", N_c, " μ: ", μ)
	μ = tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	t += 1
	state_new = ESState(x_new, μ, t)
	return ESSample(state_new.x), state_new
end


In [19]:
rng = Random.default_rng(89)
ndims = 10
n_walkers = 80#2 * ndims
sampler = EnsembleSliceSampler(1.0, 50, n_walkers, 10_000)

# initialize the walkers
m = AbstractMCMC.LogDensityModel(MYmodel).logdensity
init_start = randn(rng, n_walkers, ndims)
init_start[:,2] = abs.(init_start[:,2])
logdens = [ LogDensityProblems.logdensity(m,init_start[i,:]) for i in 1:n_walkers]
@assert all(logdens .!= -Inf) "Initial positions are not valid"

state = ESState(init_start, 1.0, 1)

ESState{Matrix{Float64}, Float64, Int64}([-1.5857037467258344 0.880573792808148 … -1.4586472930307777 -1.354915157614762; -1.0926851772841528 1.1254576170709583 … -0.3086706631668927 -0.28691971540571837; … ; 1.3785271775583172 0.700844109549022 … -1.2064673046448453 -0.9395191976002133; 0.5960501297430083 0.3487050830205569 … 1.2730625124001318 -0.9539916677732265], 1.0, 1)

In [20]:
rng = Random.default_rng(2)

x_next, state_next = AbstractMCMC.step(
    rng,
    AbstractMCMC.LogDensityModel(MYmodel),
    sampler,
    state
)

(ESSample{Matrix{Float64}}([-1.4349608938236587 1.6706820892795795 … -2.0180947685847923 -0.048520377599373; -0.9197694267205594 0.9844422386963714 … -0.2214744411543719 0.11112414766699247; … ; 1.0007393765429593 0.4152680021386759 … -1.2931804559219668 -1.0524668212289485; 0.6135935722167458 0.3527123528638604 … 1.2221593206577228 -0.9443177866802281]), ESState{Matrix{Float64}, Float64, Int64}([-1.4349608938236587 1.6706820892795795 … -2.0180947685847923 -0.048520377599373; -0.9197694267205594 0.9844422386963714 … -0.2214744411543719 0.11112414766699247; … ; 1.0007393765429593 0.4152680021386759 … -1.2931804559219668 -1.0524668212289485; 0.6135935722167458 0.3527123528638604 … 1.2221593206577228 -0.9443177866802281], 0.8117647058823529, 2))

In [21]:
samples = sample(MYmodel, sampler, 20_000; initial_state=state, progress=true)
samples_matrix = stack(sample -> sample.x, samples);
samples_matrix = permutedims(samples_matrix, [3, 2,1]);

Sampling   0%|                                          |  ETA: N/A
Sampling   0%|▎                                         |  ETA: 0:00:03
Sampling   1%|▍                                         |  ETA: 0:00:03
Sampling   2%|▋                                         |  ETA: 0:00:04
Sampling   2%|▉                                         |  ETA: 0:00:05
Sampling   2%|█                                         |  ETA: 0:00:05
Sampling   3%|█▎                                        |  ETA: 0:00:05
Sampling   4%|█▌                                        |  ETA: 0:00:05
Sampling   4%|█▋                                        |  ETA: 0:00:05
Sampling   4%|█▉                                        |  ETA: 0:00:05
Sampling   5%|██▏                                       |  ETA: 0:00:05
Sampling   6%|██▎                                       |  ETA: 0:00:05
Sampling   6%|██▌                                       |  ETA: 0:00:05
Sampling   6%|██▊                                       |  ETA: 0:00

In [18]:
chn = Chains(samples_matrix, ["μ", "τ", "θ[1]", "θ[2]", "θ[3]", "θ[4]", "θ[5]", "θ[6]", "θ[7]", "θ[8]"])

Chains MCMC chain (20000×10×250 Array{Float64, 3}):

Iterations        = 1:1:20000
Number of chains  = 250
Samples per chain = 20000
parameters        = μ, τ, θ[1], θ[2], θ[3], θ[4], θ[5], θ[6], θ[7], θ[8]

Summary Statistics
  parameters      mean       std      mcse      ess_bulk      ess_tail      rh ⋯
      Symbol   Float64   Float64   Float64       Float64       Float64   Float ⋯

           μ    4.3922    3.3199    0.0072   214025.6261   372969.0180    1.00 ⋯
           τ    3.5813    3.1664    0.0093   174461.5353   129820.9739    1.00 ⋯
        θ[1]    0.3173    0.9892    0.0021   218014.4600   394369.7430    1.00 ⋯
        θ[2]    0.0974    0.9373    0.0020   220209.8316   378390.6042    1.00 ⋯
        θ[3]   -0.0850    0.9688    0.0021   220614.9817   401124.8007    1.00 ⋯
        θ[4]    0.0634    0.9441    0.0020   221754.5016   398839.9471    1.00 ⋯
        θ[5]   -0.1625    0.9333    0.0020   220795.9202   383161.2618    1.00 ⋯
        θ[6]   -0.0712    0.9438    0.0020  

# Now with turing.jl

In [10]:
function AbstractMCMC.step(
    rng::Random.AbstractRNG,
    model_wrapper::AbstractMCMC.LogDensityModel,
    ::EnsembleSliceSampler;
    kwargs...)
    println("step")
    model = model_wrapper.logdensity
    nwalkers = sampler.N_walkers
    ndims = LogDensityProblems.dimension(model)
    x = randn(rng,nwalkers,ndims)
    x[:,2] = abs.(x[:,2])
    return ESSample(x), ESState(x,1.0,1)
end

In [11]:
using Turing
@model function school_reparam(y::AbstractVector{<:Float64}, σ::AbstractVector{<:Float64}, n_schools::Int64=8)
    μ ~ Normal(0, 5)
    τ ~ truncated(Cauchy(0, 5), lower = 0)
    θ ~ filldist(Normal(0, 1), n_schools)
    for i in 1:n_schools
        y[i] ~ Normal(μ + τ * θ[i], σ[i])
    end
end

mymod = school_reparam(y, σ)
Turing.Inference.getparams(::Turing.Model, sample::ESSample) = sample.x

In [12]:
chain = sample(mymod, externalsampler(sampler), 10)

Sampling   0%|                                          |  ETA: N/A


step


Sampling 100%|██████████████████████████████████████████| Time: 0:00:06


MethodError: MethodError: no method matching unflatten(::DynamicPPL.TypedVarInfo{@NamedTuple{μ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:μ, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{AbstractPPL.VarName{:μ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, τ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:τ, typeof(identity)}, Int64}, Vector{Truncated{Cauchy{Float64}, Continuous, Float64, Float64, Nothing}}, Vector{AbstractPPL.VarName{:τ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, θ::DynamicPPL.Metadata{Dict{AbstractPPL.VarName{:θ, typeof(identity)}, Int64}, Vector{DistributionsAD.TuringScalMvNormal{Vector{Float64}, Float64}}, Vector{AbstractPPL.VarName{:θ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}}, Float64}, ::Matrix{Float64})

Closest candidates are:
  unflatten(::DynamicPPL.TypedVarInfo, !Matched::NamedTuple)
   @ Turing ~/.julia/packages/Turing/QN7BL/src/mcmc/Inference.jl:165
  unfl